In [1]:
import os
import pandas as pd

In [2]:
DATASET = "Mouse-2023"

subfolders = [
  "documentation",
  "raw-data",
  "processed-data",
  "QC-results",
  "DEA-results",
  "raw-GSEA-results",
  "GSEA-results"
]

for folder in subfolders:
  path = os.path.join(DATASET, folder)
  if not os.path.exists(path):
    os.makedirs(path)


**Make QC Summary from STAR Logs**

In [7]:
def parse_star_log(filepath):
  stats = {}
  # Extract sample ID from filename
  sample_id = os.path.basename(filepath).replace("_Log.final.out", "")
  stats['Sample'] = sample_id

  with open(filepath) as f:
    for line in f:
      if '|' in line:
        key, value = line.split('|')
        stats[key.strip()] = value.strip()

  return stats

In [ ]:
log_folder = os.path.join(DATASET, "raw-data", "STAR-logs")
all_stats = []

for filename in os.listdir(log_folder):
  if filename.endswith("Log.final.out"):
    filepath = os.path.join(log_folder, filename)
    all_stats.append(parse_star_log(filepath))

qc_df = pd.DataFrame(all_stats)

In [15]:
keep_cols = [
  'Sample',
  'Number of input reads',
  'Uniquely mapped reads number',
  'Uniquely mapped reads %',
  'Number of reads mapped to multiple loci',
  '% of reads mapped to multiple loci',
]
qc_df = qc_df[keep_cols]

out_path = os.path.join(DATASET, "raw-data", "STAR_mapping_stats.csv")
qc_df.to_csv(out_path, index=False)